# NeMo Cache-Aware Streaming — Model Output Inspector

Tests `nvidia/nemotron-speech-streaming-en-0.6b` chunk by chunk and shows exactly what `conformer_stream_step` returns.

> ⚠️ This model is **English-only**. French audio will produce hallucinated English text — that is expected behaviour, not a bug in the code.

In [ ]:
import os, time, tempfile
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import numpy as np
import soundfile as sf
import torch
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.parts.utils.streaming_utils import CacheAwareStreamingAudioBuffer

SAMPLE_RATE = 16_000
MODEL_NAME  = 'nvidia/nemotron-speech-streaming-en-0.6b'

# chunk-ms → att_context_size [left_frames, right_frames]
CHUNK_CONFIGS = {
     80: [70,  0],
    160: [70,  1],
    560: [70,  6],
   1120: [70, 13],
}

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load model

In [ ]:
CHUNK_MS = 160   # change to 80 / 560 / 1120 to experiment

t0 = time.perf_counter()
model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)
model.encoder.set_default_att_context_size(CHUNK_CONFIGS[CHUNK_MS])
model = model.to(device).eval()
print(f'Loaded in {time.perf_counter()-t0:.1f}s | att_context_size={CHUNK_CONFIGS[CHUNK_MS]}')

## 2. Load audio

Point `AUDIO_PATH` at your file, or leave `None` to generate a synthetic tone.

In [ ]:
import librosa

AUDIO_PATH = None   # e.g. '/path/to/audio.mp3' or '.wav'

_tmp = None
if AUDIO_PATH:
    audio, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True, dtype=np.float32)
    _tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(_tmp.name, audio, SAMPLE_RATE)
    _tmp.close()
    wav_path = _tmp.name
    print(f'Loaded: {len(audio)/SAMPLE_RATE:.2f}s  → temp WAV {wav_path}')
else:
    duration = 3.0
    t = np.linspace(0, duration, int(SAMPLE_RATE * duration), dtype=np.float32)
    audio = (0.3 * np.sin(2 * np.pi * 440.0 * t)).astype(np.float32)
    _tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(_tmp.name, audio, SAMPLE_RATE)
    _tmp.close()
    wav_path = _tmp.name
    print(f'Synthetic tone: {duration}s → {wav_path}')

## 3. Stream and collect raw Hypothesis objects

In [ ]:
cache_last_channel, cache_last_time, cache_last_channel_len = (
    model.encoder.get_initial_cache_state(batch_size=1)
)
previous_hypotheses = None

buf = CacheAwareStreamingAudioBuffer(model=model, online_normalization=True)
buf.append_audio_file(wav_path, stream_id=-1)

rows = []   # one row per chunk

for step, (chunk_audio, chunk_lengths) in enumerate(buf):
    chunk_audio   = chunk_audio.to(device)
    chunk_lengths = chunk_lengths.to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        (pred_out, step_texts,
         cache_last_channel, cache_last_time, cache_last_channel_len,
         previous_hypotheses) = model.conformer_stream_step(
            processed_signal=chunk_audio,
            processed_signal_length=chunk_lengths,
            cache_last_channel=cache_last_channel,
            cache_last_time=cache_last_time,
            cache_last_channel_len=cache_last_channel_len,
            keep_all_outputs=buf.is_buffer_empty(),
            previous_hypotheses=previous_hypotheses,
            drop_extra_pre_encoded=0,
        )
    elapsed_ms = (time.perf_counter() - t0) * 1000

    hyp  = step_texts[0] if step_texts else None
    text = hyp.text if (hyp is not None and hyp.text) else ''
    score = float(hyp.score) if hyp is not None else float('nan')

    rows.append({'step': step + 1, 'text': text, 'score': score, 'latency_ms': elapsed_ms})

    marker = '▶' if text else ' '
    print(f'  {marker} step {step+1:4d}  [{elapsed_ms:5.1f}ms]  score={score:7.2f}  {text!r}')

# Cleanup temp file
if _tmp:
    os.unlink(wav_path)

print(f'\nTotal steps: {len(rows)}')

## 4. Inspect the Hypothesis object fields

In [ ]:
# Show all fields of the last non-empty Hypothesis
if previous_hypotheses:
    hyp = previous_hypotheses[0]
    print(type(hyp))
    for attr in dir(hyp):
        if attr.startswith('_'):
            continue
        val = getattr(hyp, attr)
        if callable(val):
            continue
        print(f'  {attr:30s}: {val}')
else:
    print('No hypotheses produced.')

## 5. Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

text_rows = df[df['text'] != '']
final_transcript = text_rows['text'].iloc[-1] if not text_rows.empty else ''

print('─' * 60)
print(f"Final transcript : {final_transcript!r}")
print(f"Steps with text  : {len(text_rows)} / {len(df)}")
print(f"Avg latency      : {df['latency_ms'].mean():.1f}ms  "
      f"p95={df['latency_ms'].quantile(.95):.1f}ms  max={df['latency_ms'].max():.1f}ms")
print(f"Real-time factor : {df['latency_ms'].mean() / CHUNK_MS:.3f}x")
print()
print('Note: English-only model on French audio → hallucinated output is expected.')
print('For French, try: nvidia/parakeet-tdt-1.1b  (multilingual) or a dedicated fr model.')

df.tail(20)